# 00 — Install & Quickstart

**TAPLite4MPO** is not just an assignment engine — it is a reproducible MPO workflow: *data → validate → run → report*, all sitting on a **no-guessing intake gate** that refuses to run on undeclared data conventions.

This notebook: install, confirm the kernel is built, and run one assignment end-to-end in five lines with the Python API.

> Run these notebooks from the **repository root** so the relative paths resolve.

## 1. Install
```bash
pip install -e .            # from the repo root (editable)
# optional extras:
pip install -e '.[runconfig]'   # pyyaml for `taplite run <config.yml>`
```
The Python layer is stdlib-only; the **C++ kernel is the solver** and must be built separately (`cmake` / `build.sh` → `DTALite_exe.exe`).

In [1]:
import os, sys
# ensure repo root on sys.path when running in-place (editable install not required)
ROOT = os.getcwd()
assert os.path.exists('dtalite_qa'), 'run this notebook from the repo root'
# platform-aware kernel lookup: bin/ first (bash build.sh), then the raw CMake output
EXE = next((p for p in ('bin/DTALite.exe', 'bin/DTALite',
                        'cmake_build_rel/DTALite_exe.exe', 'cmake_build_rel/DTALite_exe')
            if os.path.exists(p)), 'bin/DTALite.exe')
print('kernel exe present:', os.path.exists(EXE))

kernel exe present: True


## 2. The two front doors
- **Python API** (developers): `Network / Demand / Scenario / AssignmentEngine / Result`.
- **CLI** (analysts): `taplite validate | run | report | compare`.

Both inherit the intake gate. Quickstart uses the API.

In [2]:
from dtalite_qa.api import Network, Demand, Scenario, AssignmentEngine
net = Network.read_gmns('kernel/data_sets/03_chicago_sketch')
net

<Network 03_chicago_sketch: 933 nodes, 2950 links, 387 zones>

In [3]:
scen = Scenario(net, Demand.from_network(net), settings={'iterations': 20})
result = AssignmentEngine().run(scen, exe=EXE)
print('converged gap %:', result.final_gap_pct())
result.moe()

converged gap %: 0.19062


{'links': 2950,
 'loaded_links': 2928,
 'vmt': 36499904.9,
 'vht': 919678.8,
 'mean_speed_mph': 39.69}

## 3. One-command report
Turn the finished run into a single self-contained HTML (no CDN, opens offline).

In [4]:
from dtalite_qa import report_html
html = report_html.build_report(result.run_dir, project_name='Chicago Sketch quickstart')
print('report ->', html)

report -> C:\Users\xzhou\AppData\Local\Temp\dtalite_run_6vlxjr6n\report.html


## 4. The gate is not optional
A scenario that has not **declared** its conventions (capacity basis/period, PLF, units, trip kind) refuses to run. That discipline is the whole product — a prettier way to get a *wrong* answer faster is a net negative.

```bash
taplite validate <scenario>   # BLOCKS until submission.yml is filled and intake passes
```

Next: **01_load_gmns_network** (what node/link/zone/demand mean) and **02_baseline_assignment** (run + read the report).